<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">מחברת 2.2: ארבעת הפעלים וה-<span dir="ltr">pipe</span></p>
<p dir="rtl" style="direction:rtl; text-align:right;">זהו המשך מודרך של חלק ב במפגש 2. נתרגל את <code dir="ltr">filter()</code>, <code dir="ltr">select()</code>, <code dir="ltr">arrange()</code> ו-<code dir="ltr">mutate()</code> ואת ה-<span dir="ltr">pipe</span> (<code dir="ltr">|&gt;</code>, "ואז") על הטבלה הנקייה של 49 המטופלים אחרי שבץ. משך מוערך: כ-25 דקות. השלימו את החסר (<code dir="ltr">___</code>), הריצו, ואז הריצו את תא הבדיקה שמתחתיו. בכל תא יש פקודה אחת שמדפיסה תוצאה.</p>
<p dir="rtl" style="direction:rtl; text-align:right;"><b>כך עובד כל תא בדיקה:</b> אם התשובה נכונה יודפס <code dir="ltr">OK</code>. אם לא, תופיע שגיאה, וזה בסדר: תקנו והריצו שוב.</p>
<p dir="rtl" style="direction:rtl; text-align:right;">עמודות הטבלה: <code dir="ltr">id</code>, <code dir="ltr">sex</code> (<code dir="ltr">F</code>/<code dir="ltr">M</code>), <code dir="ltr">age</code>, <code dir="ltr">lesion_side</code> (<code dir="ltr">L</code>/<code dir="ltr">R</code>), <code dir="ltr">lesion_type</code> (<code dir="ltr">ischemic</code>/<code dir="ltr">hemorrhagic</code>/<code dir="ltr">mixed</code>), <code dir="ltr">lesion_vol</code>, <code dir="ltr">fma_sum_sub</code> (<span dir="ltr">FMA-UE</span> בשלב התת-חריף, 0 עד 66), <code dir="ltr">fma_sum_chr</code> (בשלב הכרוני), <code dir="ltr">tao_sub</code> ו-<code dir="ltr">tao_chr</code> (ימים מההתחלה) ועוד.</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנת הסביבה</p>
<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא מתקין רק את החבילות הדרושות, ממאגר בינארי מהיר, ומדפיס כמה זמן ההתקנה נמשכה. הריצה הראשונה אורכת זמן מה.</p>

In [ ]:
# Install only the packages the course needs, from a binary repository (fast)
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
needed  <- c("tidyverse")          # add packages here when a notebook needs them
missing <- setdiff(needed, rownames(installed.packages()))   # skip installed ones
t <- system.time(if (length(missing) > 0) install.packages(missing))
cat("Installed:", if (length(missing)) paste(missing, collapse = ", ") else "nothing new",
    "| elapsed seconds:", round(t[["elapsed"]], 1), "\n")
library(tidyverse)

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">טעינת נתונים</p>

<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא טוען את הפונקציה <code dir="ltr">load_course_data()</code> ממאגר הקורס ב-<span dir="ltr">GitHub</span>. הפקודה <code dir="ltr">load_course_data("stroke_clean")</code> מחזירה את הטבלה הנקייה (אחרי הניקוי שבפרק 5), ו-<code dir="ltr">as_tibble()</code> הופכת אותה לטיבל.</p>

In [ ]:
source("https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/R/load_course_data.R")
stroke <- as_tibble(load_course_data("stroke_clean"))   # the cleaned table, 49 patients

In [ ]:
glimpse(stroke)

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 1: <code dir="ltr">filter()</code></p>

<p dir="rtl" style="direction:rtl; text-align:right;">השאירו רק מטופלים עם שבץ המורגי (<code dir="ltr">hemorrhagic</code>) שגילם קטן מ-60. זכרו: שוויון נכתב <code dir="ltr">==</code>, טקסט בגרשיים, ופסיק בין תנאים פירושו "וגם". שמרו ב-<code dir="ltr">young_h</code>.</p>

In [ ]:
young_h <- stroke |>
  filter(lesion_type == ___,   # TODO: the text value (in quotation marks)
         age ___ 60)           # TODO: the comparison operator
young_h |> select(id, age, lesion_type)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(young_h) == 9, all(young_h$lesion_type == "hemorrhagic"), all(young_h$age < 60))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 2: <code dir="ltr">select()</code> ו-<code dir="ltr">arrange()</code></p>

<p dir="rtl" style="direction:rtl; text-align:right;"><code dir="ltr">select()</code> בוחרת עמודות. <code dir="ltr">arrange()</code> ממיינת את השורות לפי עמודה, וכברירת מחדל <b>בסדר עולה</b>: מהערך הקטן לגדול. כדי למיין <b>בסדר יורד</b> (מהגדול לקטן) עוטפים את שם העמודה ב-<code dir="ltr">desc()</code>: <code dir="ltr">arrange(desc(age))</code>.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>2א. סדר עולה (ברירת המחדל).</b> הציגו את העמודות <code dir="ltr">id</code>, <code dir="ltr">age</code>, <code dir="ltr">fma_sum_chr</code> ממוינות מהציון הכרוני הנמוך לגבוה, ושמרו את שלוש השורות הראשונות ב-<code dir="ltr">bottom3</code>.</p>

In [ ]:
bottom3 <- stroke |>
  select(id, age, ___) |>           # TODO: keep id, age and fma_sum_chr
  arrange(___) |>                   # TODO: sort by fma_sum_chr (ascending is the default)
  head(3)                           # first three rows
bottom3

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(identical(names(bottom3), c("id", "age", "fma_sum_chr")), nrow(bottom3) == 3,
          bottom3$fma_sum_chr[1] == min(stroke$fma_sum_chr))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>2ב. סדר יורד.</b> אותו דבר, אבל מהציון הכרוני הגבוה לנמוך: הוסיפו <code dir="ltr">desc()</code>. שמרו ב-<code dir="ltr">top3</code>.</p>

In [ ]:
top3 <- stroke |>
  select(id, age, fma_sum_chr) |>   # keep three columns
  arrange(___(fma_sum_chr)) |>      # TODO: the function that reverses the order
  head(3)                           # first three rows
top3

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(identical(names(top3), c("id", "age", "fma_sum_chr")), nrow(top3) == 3,
          all(top3$fma_sum_chr == 66))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 3: <code dir="ltr">mutate()</code> (עמודה חדשה)</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><code dir="ltr">mutate()</code> מוסיפה עמודה שמחושבת מעמודות אחרות, שורה אחרי שורה. נעשה זאת בשני צעדים.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>3א. עמודה מספרית.</b> חשבו לכל מטופל את השינוי בציון: הציון הכרוני פחות התת-חריף, בעמודה <code dir="ltr">fma_change</code>. שמרו ב-<code dir="ltr">stroke_chg</code>.</p>

In [ ]:
stroke_chg <- stroke |>
  mutate(fma_change = ___ - ___)   # TODO: chronic minus subacute score
stroke_chg |> select(id, fma_sum_sub, fma_sum_chr, fma_change) |> head(3)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(all(stroke_chg$fma_change == stroke$fma_sum_chr - stroke$fma_sum_sub))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>3ב. עמודה לוגית.</b> הוסיפו ל-<code dir="ltr">stroke_chg</code> עמודה <code dir="ltr">improved</code> שהיא <code dir="ltr">TRUE</code> כשהשיפור הוא 10 נקודות או יותר (בקירוב השינוי המינימלי החשוב קלינית). הממוצע של עמודה לוגית הוא שיעור ה-<code dir="ltr">TRUE</code>: <code dir="ltr">mean()</code> נותן את חלק המטופלים שהשתפרו.</p>

In [ ]:
stroke_chg <- stroke_chg |>
  mutate(improved = fma_change ___ 10)   # TODO: TRUE if the change is 10 or more
round(mean(stroke_chg$improved), 2)      # proportion of TRUE

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(sum(stroke_chg$improved) == 15)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 4: סוגים נוספים של <code dir="ltr">mutate()</code></p>

<p dir="rtl" style="direction:rtl; text-align:right;"><code dir="ltr">mutate()</code> יכולה גם לדרג וגם לחלק לקבוצות. שתי הפונקציות האלה עובדות בתוך <code dir="ltr">mutate()</code>, ונתרגל כל אחת בנפרד.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>אפשרות 1: דירוג עם <code dir="ltr">min_rank()</code>.</b> נותנת לכל ערך את מקומו. ערכים שווים מקבלים אותו מקום. כברירת מחדל הערך הקטן ביותר הוא מקום 1; עם <code dir="ltr">desc()</code> הערך הגבוה ביותר הוא מקום 1.</p>

In [ ]:
min_rank(c(50, 30, 50, 10))        # the smallest value (10) is rank 1; the two 50s share a rank

In [ ]:
min_rank(desc(c(50, 30, 50, 10)))  # with desc(): the largest value is rank 1

<p dir="rtl" style="direction:rtl; text-align:right;"><b>4א.</b> הוסיפו עמודה <code dir="ltr">rank_chr</code> שבה הציון הכרוני הגבוה ביותר הוא מקום 1. שמרו ב-<code dir="ltr">stroke_rank</code>.</p>

In [ ]:
stroke_rank <- stroke |>
  mutate(rank_chr = min_rank(___(fma_sum_chr)))   # TODO: reverse the order so that rank 1 = highest score
stroke_rank |> select(id, fma_sum_chr, rank_chr) |> arrange(rank_chr) |> head(3)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(min(stroke_rank$rank_chr) == 1, stroke_rank$rank_chr[stroke_rank$fma_sum_chr == 66][1] == 1)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>אפשרות 2: חלוקה לקטגוריות עם <code dir="ltr">cut()</code>.</b> מחלקת מספרים לקבוצות. <code dir="ltr">breaks</code> הם הגבולות (<code dir="ltr">-Inf</code> ו-<code dir="ltr">Inf</code> = מינוס ופלוס אינסוף) ו-<code dir="ltr">labels</code> הם השמות, אחד לכל קבוצה. כל קבוצה כוללת את הגבול העליון שלה: 28 שייך לקבוצה הראשונה.</p>

In [ ]:
cut(c(5, 30, 60),                           # three example scores
    breaks = c(-Inf, 28, 42, Inf),            # edges: up to 28, 29-42, above 42
    labels = c("severe", "moderate", "mild")) # one name per group

<p dir="rtl" style="direction:rtl; text-align:right;"><b>4ב.</b> נשתמש בחלוקה של <span dir="ltr">Woodbury</span> ועמיתיה (2013): ליקוי חמור (0 עד 28), בינוני (29 עד 42) וקל (43 עד 66). הוסיפו עמודה <code dir="ltr">severity_sub</code> לפי הציון התת-חריף. שמרו ב-<code dir="ltr">stroke_sev</code>.</p>

In [ ]:
stroke_sev <- stroke |>
  mutate(severity_sub = cut(fma_sum_sub,
                            breaks = c(-Inf, 28, ___, Inf),   # TODO: the edge between moderate and mild
                            labels = c("severe", "moderate", "mild")))
stroke_sev |> select(id, fma_sum_sub, severity_sub) |> head(3)

<p dir="rtl" style="direction:rtl; text-align:right;"><b>4ג.</b> <code dir="ltr">count()</code> תספור כמה מטופלים בכל דרגה:</p>

In [ ]:
stroke_sev |> count(severity_sub)   # patients per severity group

In [ ]:
# Visible check: stops with an error if the answer is wrong
sev_n <- table(stroke_sev$severity_sub)
stopifnot(sev_n[["severe"]] == 18, sev_n[["moderate"]] == 5, sev_n[["mild"]] == 26)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 5: משפט אחד עם <code dir="ltr">|&gt;</code></p>

<p dir="rtl" style="direction:rtl; text-align:right;">את ארבעת הפעלים מחברים ל<b>משפט אחד</b> שנקרא בקול "ואז". נבנה אותו צעד אחר צעד. הנה הפעלים, לפי הסדר שבו נשתמש בהם, עם דוגמה לכל אחד:</p>

<ol dir="rtl" style="direction:rtl; text-align:right;">
<li><code dir="ltr">filter()</code>: שומרת שורות לפי תנאי. למשל <code dir="ltr">filter(age &gt;= 65)</code>.</li>
<li><code dir="ltr">mutate()</code>: מוסיפה עמודה מחושבת. למשל <code dir="ltr">mutate(fma_change = fma_sum_chr - fma_sum_sub)</code>.</li>
<li><code dir="ltr">select()</code>: שומרת עמודות. למשל <code dir="ltr">select(id, age)</code>.</li>
<li><code dir="ltr">arrange()</code>: ממיינת שורות. למשל <code dir="ltr">arrange(desc(age))</code>.</li>
</ol>
<p dir="rtl" style="direction:rtl; text-align:right;">המטרה: גברים (<code dir="ltr">M</code>) עם נגע בצד שמאל (<code dir="ltr">L</code>) בני 65 ומעלה, עם עמודת שינוי, שלוש עמודות בלבד, ממוינים מהשיפור הגדול לקטן. בכל צעד מוסיפים שורה אחת לצינור שבונים ב-<code dir="ltr">men_left</code>. בצעד האחרון הצינור מלא.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>5א. צעד 1: שומרים את השורות.</b> גברים, נגע שמאלי, גיל 65 ומעלה.</p>

In [ ]:
men_left <- stroke |>
  filter(sex == "M", lesion_side == "L", age ___ 65)   # TODO: age 65 or older
men_left |> select(id, sex, lesion_side, age)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(is.data.frame(men_left), nrow(men_left) == 6, all(men_left$sex == "M"))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>5ב. צעד 2: מוסיפים עמודה.</b> הוסיפו אחרי ה-<code dir="ltr">filter()</code> את <code dir="ltr">|&gt; mutate()</code> שמחשב את <code dir="ltr">fma_change</code>.</p>

In [ ]:
men_left <- stroke |>
  filter(sex == "M", lesion_side == "L", age >= 65) |>   # step 1
  mutate(___)                                              # TODO step 2: fma_change = chronic - subacute
men_left |> select(id, age, fma_change)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot("fma_change" %in% names(men_left), nrow(men_left) == 6)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>5ג. צעד 3: בוחרים עמודות.</b> הוסיפו <code dir="ltr">|&gt; select()</code> שמשאיר רק <code dir="ltr">id</code>, <code dir="ltr">age</code>, <code dir="ltr">fma_change</code>.</p>

In [ ]:
men_left <- stroke |>
  filter(sex == "M", lesion_side == "L", age >= 65) |>   # step 1
  mutate(fma_change = fma_sum_chr - fma_sum_sub) |>       # step 2
  select(___)                                             # TODO step 3: id, age, fma_change
men_left

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(identical(names(men_left), c("id", "age", "fma_change")))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>5ד. צעד 4: ממיינים.</b> הוסיפו <code dir="ltr">|&gt; arrange()</code> מהשיפור הגדול לקטן.</p>

In [ ]:
men_left <- stroke |>
  filter(sex == "M", lesion_side == "L", age >= 65) |>   # step 1
  mutate(fma_change = fma_sum_chr - fma_sum_sub) |>       # step 2
  select(id, age, fma_change) |>                          # step 3
  arrange(___)                                            # TODO step 4: biggest improvement first
men_left

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(is.data.frame(men_left), nrow(men_left) == 6, men_left$id[1] == 1063,
          identical(names(men_left), c("id", "age", "fma_change")))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 6: <code dir="ltr">rename()</code> ו-<code dir="ltr">case_match()</code></p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>6א. שינוי שמות עמודות.</b> <code dir="ltr">rename()</code> משנה שם של עמודה, בתחביר <code dir="ltr">שם_חדש = שם_ישן</code>. החליפו את השמות <code dir="ltr">tao_sub</code> ו-<code dir="ltr">tao_chr</code> ל-<code dir="ltr">days_sub</code> ו-<code dir="ltr">days_chr</code>.</p>

In [ ]:
stroke_rn <- stroke |>
  rename(days_sub = ___,           # TODO: new name = old name (tao_sub)
         days_chr = tao_chr)
names(stroke_rn)[7:9]

<p dir="rtl" style="direction:rtl; text-align:right;"><b>6ב. שינוי ערכים בתוך עמודה.</b> <code dir="ltr">case_match()</code> משנה ערכים: כל שורה היא <code dir="ltr">"ערך_ישן" ~ "ערך_חדש"</code> (הסימן <code dir="ltr">~</code> נקרא "הופך ל-"). ערך שלא הוגדר לו כלל הופך ל-<code dir="ltr">NA</code>. החליפו את הקודים <code dir="ltr">F</code> ו-<code dir="ltr">M</code> ב-<code dir="ltr">female</code> ו-<code dir="ltr">male</code>. <b>שימו לב:</b> כשתריצו את התא תופיע אזהרה בצבע אדום או כתום. זה בכוונה! נסביר אותה מיד.</p>

In [ ]:
stroke_rn <- stroke_rn |>
  mutate(sex = case_match(sex,
                          "F" ~ "female",   # old value ~ new value
                          "M" ~ ___))       # TODO: the new value for "M"

<p dir="rtl" style="direction:rtl; text-align:right; border-right:4px solid #d98e04; background:#fff6e0; padding:6px 14px; margin:0;"><b>מה זו אזהרת הוצאה משימוש (<span dir="ltr">deprecation warning</span>)?</b></p>
<p dir="rtl" style="direction:rtl; text-align:right; border-right:4px solid #d98e04; background:#fff6e0; padding:6px 14px; margin:0;">האזהרה אומרת ש-<code dir="ltr">case_match()</code> <b>הוצאה משימוש</b> (<span dir="ltr">deprecated</span>) בגרסה <span dir="ltr">dplyr 1.2.0</span>: המפתחים החליטו להחליף אותה בפונקציה חדשה. <b>זו אזהרה, לא שגיאה:</b> הקוד ממשיך לרוץ והתוצאה נכונה (בדקו!). אבל בגרסאות עתידיות הפונקציה הישנה עלולה להיעלם, ולכן כדאי להתרגל לפונקציה החדשה: <code dir="ltr">recode_values()</code> (קיימת מ-<span dir="ltr">dplyr 1.2.0</span> ומעלה). מה עושים כשרואים אזהרה כזאת? קוראים אותה עד הסוף (השורה "<span dir="ltr">Please use</span>" אומרת מה להחליף), בודקים שהתוצאה תקינה, ומעדכנים את הקוד כשיש זמן. אזהרות הן חלק רגיל מהעבודה עם <span dir="ltr">R</span>: לא לפחד מהן, אבל גם לא להתעלם מהן.</p>

<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא עושה את אותו דבר עם הפונקציה החדשה, בלי אזהרה:</p>

In [ ]:
stroke |>
  mutate(sex = recode_values(sex,
                             "F" ~ "female",   # the same old ~ new syntax
                             "M" ~ "male")) |>
  count(sex)

<p dir="rtl" style="direction:rtl; text-align:right;">ובדיקת התוצאה של <code dir="ltr">case_match()</code>: כמה גברים ונשים, וערך חסר אחד.</p>

In [ ]:
stroke_rn |> count(sex)   # the missing sex stays NA

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(all(c("days_sub", "days_chr") %in% names(stroke_rn)), !"tao_sub" %in% names(stroke_rn),
          sum(stroke_rn$sex == "male", na.rm = TRUE) == 37, sum(is.na(stroke_rn$sex)) == 1)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 7: שגיאה לאבחון</p>

<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא מריץ בכוונה צינור שנכשל, ו-<code dir="ltr">tryCatch()</code> מדפיס את ההודעה במקום לעצור את הריצה. קראו אותה: ההודעה אומרת ש-<span dir="ltr">R</span> לא מוצא את העמודה <code dir="ltr">fma_sum_chr</code>, אבל העמודה קיימת בטבלה! מה קרה בין השורות? (רמז: מה עושה <code dir="ltr">select()</code> לטבלה שממשיכה הלאה ב-<code dir="ltr">|&gt;</code>?) בתא שאחריו תקנו את הצינור (בלי להוסיף שורות) ושמרו את התוצאה ב-<code dir="ltr">fixed</code>.</p>

In [ ]:
# Run on purpose a pipeline that fails and print the error message
tryCatch(
  stroke |>
    select(id, age) |>                                    # keep two columns
    mutate(fma_change = fma_sum_chr - fma_sum_sub),      # use two other columns
  error = function(e) cat("Error message:", cli::ansi_strip(conditionMessage(e)), "\n"))

In [ ]:
# TODO: fix the pipeline above (hint: change only the select() line)
fixed <- NULL

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(is.data.frame(fixed), "fma_change" %in% names(fixed), nrow(fixed) == 49)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 8: <code dir="ltr">pivot_longer()</code></p>

<p dir="rtl" style="direction:rtl; text-align:right;">לטבלה "רחבה" יש עמודה לכל שלב (<span dir="ltr">phase</span>): <code dir="ltr">fma_sum_sub</code>, <code dir="ltr">fma_sum_chr</code>. לטבלה "ארוכה" יש שורה לכל מטופל ושלב: נוח לגרפים. <code dir="ltr">pivot_longer()</code> מעבירה את שמות העמודות לעמודה <code dir="ltr">phase</code> ואת הציונים לעמודה <code dir="ltr">fma_sum</code>. <code dir="ltr">-id</code> פירושו "כל העמודות חוץ מ-id". <code dir="ltr">count(phase)</code> תראה כמה שורות בכל שלב.</p>

In [ ]:
stroke_long <- stroke |>
  select(id, fma_sum_sub, fma_sum_chr) |>        # wide: a column per phase
  pivot_longer(-id,                              # keep id, pivot the other columns
               names_to = "___",                 # TODO: column names go into "phase"
               values_to = "___")                # TODO: the scores go into "fma_sum"
stroke_long |> count(phase)                      # 49 rows per phase

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(stroke_long) == 98, identical(names(stroke_long), c("id", "phase", "fma_sum")))
cat("OK\n")